In [30]:
# ==============================================================================
# COMBINED TRAINING SCRIPT (PLANTVILLAGE + PLANTDOC MERGED)
# ==============================================================================
import os
import copy
import json
import shutil
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, models, transforms
from PIL import Image
from google.colab import files

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Active Device: {device}")

Active Device: cuda:0


In [32]:
# ------------------------------------------------------------------------------
# 2. Merge Cropped PlantDoc Real-World Images into PlantVillage Classes
# ------------------------------------------------------------------------------
name_mapping = {
    "Apple_Scab_Leaf": "Apple___Apple_scab",
    "Apple_leaf": "Apple___healthy",
    "Apple_rust_leaf": "Apple___Cedar_apple_rust",
    "Bell_pepper_leaf_spot": "Pepper,_bell___Bacterial_spot",
    "Bell_pepper_leaf": "Pepper,_bell___healthy",
    "Blueberry_leaf": "Blueberry___healthy",
    "Cherry_leaf": "Cherry_(including_sour)___healthy",
    "Corn_Gray_leaf_spot": "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot",
    "Corn_leaf_blight": "Corn_(maize)___Northern_Leaf_Blight",
    "Corn_rust_leaf": "Corn_(maize)___Common_rust_",
    "Peach_leaf": "Peach___healthy",
    "Potato_leaf_early_blight": "Potato___Early_blight",
    "Potato_leaf_late_blight": "Potato___Late_blight",
    "Potato_leaf": "Potato___healthy",
    "Raspberry_leaf": "Raspberry___healthy",
    "Soyabean_leaf": "Soybean___healthy",
    "Squash_Powdery_mildew_leaf": "Squash___Powdery_mildew",
    "Strawberry_leaf": "Strawberry___healthy",
    "Tomato_Early_blight_leaf": "Tomato___Early_blight",
    "Tomato_Septoria_leaf_spot": "Tomato___Septoria_leaf_spot",
    "Tomato_leaf_bacterial_spot": "Tomato___Bacterial_spot",
    "Tomato_leaf_late_blight": "Tomato___Late_blight",
    "Tomato_leaf_mosaic_virus": "Tomato___Tomato_mosaic_virus",
    "Tomato_leaf_yellow_virus": "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
    "Tomato_leaf": "Tomato___healthy",
    "Tomato_mold_leaf": "Tomato___Leaf_Mold",
    "Tomato_two_spotted_spider_mites_leaf": "Tomato___Spider_mites Two-spotted_spider_mite",
    "grape_leaf_black_rot": "Grape___Black_rot",
    "grape_leaf": "Grape___healthy"
}

pd_crop_base = "datasets/plantdoc_cropped"
merged_count = 0

if os.path.exists(pd_crop_base):
    for split in ['train', 'val']:
        split_dir = os.path.join(pd_crop_base, split)
        if not os.path.exists(split_dir):
            continue
        for pd_cls in os.listdir(split_dir):
            if pd_cls in name_mapping:
                target_pv_cls = name_mapping[pd_cls]
                target_dir = os.path.join(PLANTVILLAGE_DIR, target_pv_cls)
                if os.path.exists(target_dir):
                    src_cls_dir = os.path.join(split_dir, pd_cls)
                    for f in os.listdir(src_cls_dir):
                        src_file = os.path.join(src_cls_dir, f)
                        dst_file = os.path.join(target_dir, f"pd_{f}")
                        if not os.path.exists(dst_file):
                            shutil.copy(src_file, dst_file)
                            merged_count += 1
    print(f"Merged {merged_count} field images into PlantVillage classes.")

Merged 8783 field images into PlantVillage classes.


In [33]:
# ------------------------------------------------------------------------------
# 3. Data Transformations & Loaders
# ------------------------------------------------------------------------------
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

class TransformWrapper(torch.utils.data.Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform
    def __getitem__(self, idx):
        x, y = self.subset[idx]
        return self.transform(x), y
    def __len__(self):
        return len(self.subset)

combined_raw = datasets.ImageFolder(PLANTVILLAGE_DIR)
all_classes = combined_raw.classes
num_classes = len(all_classes)
print(f"Total Combined Dataset: {len(combined_raw)} images across {num_classes} classes.")

train_len = int(0.85 * len(combined_raw))
val_len = len(combined_raw) - train_len
train_sub, val_sub = random_split(combined_raw, [train_len, val_len], generator=torch.Generator().manual_seed(42))

loaders = {
    'train': DataLoader(TransformWrapper(train_sub, train_transform), batch_size=64, shuffle=True, num_workers=2, pin_memory=True),
    'val': DataLoader(TransformWrapper(val_sub, val_transform), batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
}
dataset_sizes = {'train': train_len, 'val': val_len}

Total Combined Dataset: 63089 images across 38 classes.


In [34]:
# ------------------------------------------------------------------------------
# 4. Model Setup
# ------------------------------------------------------------------------------
model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
in_features = model.classifier[3].in_features
model.classifier[3] = nn.Linear(in_features, num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.5)

In [35]:
# ------------------------------------------------------------------------------
# 5. Training Loop
# ------------------------------------------------------------------------------
num_epochs = 6
best_w = copy.deepcopy(model.state_dict())
best_acc = 0.0

print(f"\nTraining on combined dataset on {device}...")
for epoch in range(num_epochs):
    print(f"Epoch {epoch + 1}/{num_epochs}")
    for phase in ['train', 'val']:
        model.train() if phase == 'train' else model.eval()
        running_loss, running_corrects = 0.0, 0

        for inputs, labels in loaders[phase]:
            inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            optimizer.zero_grad()

            with torch.set_grad_enabled(phase == 'train'):
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)

                if phase == 'train':
                    loss.backward()
                    optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)

        if phase == 'train':
            scheduler.step()

        ep_loss = running_loss / dataset_sizes[phase]
        ep_acc = (running_corrects.double() / dataset_sizes[phase]).item()
        print(f"  {phase.upper():<5} Loss: {ep_loss:.4f} | Acc: {ep_acc * 100:.2f}%")

        if phase == 'val' and ep_acc > best_acc:
            best_acc = ep_acc
            best_w = copy.deepcopy(model.state_dict())

model.load_state_dict(best_w)
print(f"\nFinal Combined Validation Accuracy: {best_acc * 100:.2f}%")


Training on combined dataset on cuda:0...
Epoch 1/6
  TRAIN Loss: 0.8656 | Acc: 85.47%
  VAL   Loss: 0.6749 | Acc: 90.84%
Epoch 2/6
  TRAIN Loss: 0.6408 | Acc: 92.01%
  VAL   Loss: 0.6254 | Acc: 92.35%
Epoch 3/6
  TRAIN Loss: 0.5577 | Acc: 94.40%
  VAL   Loss: 0.5394 | Acc: 94.94%
Epoch 4/6
  TRAIN Loss: 0.5318 | Acc: 95.27%
  VAL   Loss: 0.5517 | Acc: 94.24%
Epoch 5/6
  TRAIN Loss: 0.4942 | Acc: 96.53%
  VAL   Loss: 0.5131 | Acc: 95.45%
Epoch 6/6
  TRAIN Loss: 0.4794 | Acc: 97.09%
  VAL   Loss: 0.5117 | Acc: 95.73%

Final Combined Validation Accuracy: 95.73%


In [38]:
# ------------------------------------------------------------------------------
# 6. Test Prediction (Plant and Disease Name Only)
# ------------------------------------------------------------------------------
def predict_crop_and_disease(image_path, model, class_names, min_confidence=50.0):
    model.eval()
    raw_img = Image.open(image_path).convert("RGB")
    tensor = val_transform(raw_img).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(tensor)
        probs = torch.nn.functional.softmax(outputs, dim=1)
        confidence, pred_idx = torch.max(probs, 1)

    conf_score = confidence.item() * 100.0
    if conf_score < min_confidence:
        print(f"Unrecognized (Low confidence: {conf_score:.2f}%)")
        return None

    clean_label = class_names[pred_idx.item()].replace("___", " - ").replace("_", " ")
    print(f"\nPrediction : {clean_label}")
    print(f"Confidence : {conf_score:.2f}%")
    return clean_label

if os.path.exists("/content/p1.jpeg"):
    predict_crop_and_disease("/content/WhatsApp Image 2025-09-18 at 09.51.14_1f672afb.jpg", model, all_classes)

Unrecognized (Low confidence: 19.82%)


In [89]:
from PIL import Image
import torch

def diagnose_leaf(image_path, model, class_names, min_confidence=60.0):
    model.eval()
    raw_img = Image.open(image_path).convert("RGB")
    tensor = val_transform(raw_img).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(tensor)
        probs = torch.nn.functional.softmax(outputs, dim=1)
        top_probs, top_indices = torch.topk(probs, 3, dim=1)

    top_prob = top_probs[0][0].item() * 100.0
    top_label = class_names[top_indices[0][0].item()].replace("___", " - ").replace("_", " ")

    print("=" * 60)

    # Check if confidence meets the minimum threshold
    if top_prob < min_confidence:
        print("RESULT : No leaf/plant detected or not a recognized plant image.")
        print(f"Status : Inconclusive (Confidence: {top_prob:.2f}% < {min_confidence:.0f}%)")
        print("Note   : Please upload a clear photo of a plant leaf filling the frame.")
    else:
        print(f"TOP PREDICTION : {top_label} ({top_prob:.2f}%)")
        print("=" * 50)
        print("Top 3 Candidates:")
        for i in range(3):
            idx = top_indices[0][i].item()
            p = top_probs[0][i].item() * 100.0
            lbl = class_names[idx].replace("___", " - ").replace("_", " ")
            print(f"  {i+1}. {lbl:<40} : {p:.2f}%")

    print("=" * 60)

# Run prediction
diagnose_leaf("/content/images (1).jpeg", model, all_classes, min_confidence=60.0)

TOP PREDICTION : Corn (maize) - Common rust  (98.36%)
Top 3 Candidates:
  1. Corn (maize) - Common rust               : 98.36%
  2. Apple - Cedar apple rust                 : 0.30%
  3. Corn (maize) - healthy                   : 0.12%


In [81]:
# ------------------------------------------------------------------------------
# 7. Export Delivery Deliverables for Your Teammate
# ------------------------------------------------------------------------------
print("\nExporting files for teammate...")
with open("classes.json", "w") as f:
    json.dump(all_classes, f, indent=4)

torch.save(model.state_dict(), "plant_disease_mobilenetv3.pth")

model.eval()
dummy = torch.rand(1, 3, 224, 224).to(device)
traced = torch.jit.trace(model, dummy)
traced.save("plant_disease_model_traced.pt")

print("Files saved successfully. Starting download...")
files.download("classes.json")
files.download("plant_disease_mobilenetv3.pth")
files.download("plant_disease_model_traced.pt")


Exporting files for teammate...
Files saved successfully. Starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>